<a href="https://colab.research.google.com/github/datascience-uniandes/hypothesis-testing-tutorial/blob/master/hypothesis-testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hypothesis Testing

MINE-4101: Applied Data Science
Universidad de los Andes

**Dataset:** Titanic.

Last update: September, 2026

In [56]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [39]:
ALPHA = 0.05

In [41]:
# Load the Titanic dataset from Kaggle
df = pd.read_csv("./data/titanic.csv")

In [48]:
df.shape

(891, 12)

In [50]:
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### T-test: Did survivors and non-survivors have different average ages?

In [ ]:
# BUSINESS CONTEXT:
# A shipping company wants to understand whether passenger age was related to survival during the Titanic disaster.

# H0: The mean age of survivors and non-survivors is the same.
# H1: The mean age of survivors and non-survivors is different.

survived = df.loc[df["Survived"] == 1, "Age"].dropna()
not_survived = df.loc[df["Survived"] == 0, "Age"].dropna()

t_stat, p_value = stats.ttest_ind(survived, not_survived, equal_var=False)

print("Mean survivors:", survived.mean())
print("Mean non-survivors:", not_survived.mean())
print("p-value:", p_value)

# INTERPRETATION:
# If p-value < 0.05, reject H0.
# There is statistical evidence that the two groups have different mean ages.

if p_value < ALPHA:
    print("Reject H0: mean ages are significantly different.")
else:
    print("Fail to reject H0: no significant difference was found.")

Mean survivors: 28.343689655172415
Mean non-survivors: 30.62617924528302
p-value: 0.04118965162586639
Reject H0: mean ages are significantly different.


In [44]:
# 95% CI for: mean age of survivors - mean age of non-survivors

difference = survived.mean() - not_survived.mean()

se = np.sqrt(survived.var(ddof=1) / len(survived) + not_survived.var(ddof=1) / len(not_survived))

ci = stats.t.interval(
    confidence=0.95,
    df=len(survived) + len(not_survived) - 2,
    loc=difference,
    scale=se
)

print("Difference in means:", difference)
print("95% CI:", ci)

# INTERPRETATION:
# The interval contains plausible values for the difference between the population mean ages.
# If the interval does not contain 0, this supports the conclusion that the mean ages are different.

Difference in means: -2.282489590110604
95% CI: (-4.472689529113049, -0.09228965110815812)


### ANOVA: Was the average ticket fare different across passenger classes?

In [57]:
# BUSINESS CONTEXT:
# The company wants to determine whether ticket prices differed systematically between passenger classes.

# H0: Mean fare is equal for Class 1, Class 2 and Class 3.
# H1: At least one passenger class has a different mean fare.

class_1 = df.loc[df["Pclass"] == 1, "Fare"].dropna()
class_2 = df.loc[df["Pclass"] == 2, "Fare"].dropna()
class_3 = df.loc[df["Pclass"] == 3, "Fare"].dropna()

f_stat, p_value = stats.f_oneway(class_1, class_2, class_3)

print(df.groupby("Pclass")["Fare"].mean())
print("p-value:", p_value)

# INTERPRETATION:
# If p-value < 0.05, reject H0.
# At least one passenger class has a significantly different mean fare.

# Important: ANOVA tells us that a difference exists, but not exactly which classes are different.

if p_value < ALPHA:
    print("Reject H0: at least one class has a different mean fare.")

    # 2. Post-hoc test
    # Tukey HSD tells us exactly which pairs are significantly different
    tukey = pairwise_tukeyhsd(
        endog=df["Fare"],
        groups=df["Pclass"],
        alpha=0.05
    )

    print(tukey)
else:
    print("Fail to reject H0: no significant difference was found.")

Pclass
1    84.154687
2    20.662183
3    13.675550
Name: Fare, dtype: float64
p-value: 1.0313763209141171e-84
Reject H0: at least one class has a different mean fare.
 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj  lower    upper   reject
-----------------------------------------------------
     1      2 -63.4925   0.0 -72.9165 -54.0685   True
     1      3 -70.4791   0.0 -78.1489 -62.8094   True
     2      3  -6.9866 0.108 -15.1064   1.1331  False
-----------------------------------------------------


In [47]:
for pclass, group in df.groupby("Pclass")["Fare"]:
    group = group.dropna()

    ci = stats.t.interval(
        confidence=0.95,
        df=len(group) - 1,
        loc=group.mean(),
        scale=stats.sem(group)
    )

    print(f"Class {pclass}: mean = {group.mean():.2f}, 95% CI = {ci}")

Class 1: mean = 84.15, 95% CI = (73.64281463908969, 94.6665603609103)
Class 2: mean = 20.66, 95% CI = (18.710590728785917, 22.61377557556191)
Class 3: mean = 13.68, 95% CI = (12.631171018646437, 14.719929185019549)


### Chi-squared test: Was survival associated with sex?

In [ ]:
# BUSINESS CONTEXT:
# We want to investigate whether survival and passenger sex were statistically associated.

# H0: Sex and survival are independent.
# H1: Sex and survival are associated.

table = pd.crosstab(df["Sex"], df["Survived"])

chi2, p_value, dof, expected = stats.chi2_contingency(table)

print(table)
print("p-value:", p_value)

# INTERPRETATION:
# If p-value < 0.05, reject H0.
# There is evidence of an association between sex and survival.

# Notice that this does NOT by itself establish causality.

if p_value < ALPHA:
    print("Reject H0: sex and survival are associated.")
else:
    print("Fail to reject H0: no significant association was found.")

Survived    0    1
Sex               
female     81  233
male      468  109
p-value: 1.1973570627755645e-58
Reject H0: sex and survival are associated.


### A case where we fail to reject H0

In [53]:
# BUSINESS CONTEXT:
# The company assumes that the typical passenger is about 30 years old. We want to check whether the data provides evidence against this assumption.

# H0: The mean passenger age is 30 years.
# H1: The mean passenger age is different from 30 years.

ages = df["Age"].dropna()

t_stat, p_value = stats.ttest_1samp(ages, popmean=30)

print("Mean age:", ages.mean())
print("p-value:", p_value)

if p_value < ALPHA:
    print("Reject H0: mean age is significantly different from 30.")
else:
    print("Fail to reject H0: there is not enough evidence that mean age differs from 30.")

Mean age: 29.69911764705882
p-value: 0.5801231230388639
Fail to reject H0: there is not enough evidence that mean age differs from 30.


In [55]:
ci = stats.t.interval(
    confidence=0.95,
    df=len(ages) - 1,
    loc=ages.mean(),
    scale=stats.sem(ages)
)

print("Mean age:", ages.mean())
print("95% CI:", ci)

# INTERPRETATION:
# If 30 is inside the 95% confidence interval, this is consistent with not rejecting H0 at alpha = 0.05.

# Important:
# "Fail to reject H0" does NOT mean that H0 has been proven true. It means that the data does not provide sufficient evidence against it.

Mean age: 29.69911764705882
95% CI: (28.631790041821507, 30.766445252296133)
